## GenBio-AI

This notebook can be used to format the foundation models which build on genbio-AI's tooling (AIDOCell and scFoundation)

### Environment setup

```bash
uv venv .genbio
source .genbio/bin/activate

uv pip install "modelgenerator==0.1.2" ipykernel "napistu-torch>=0.3.8"
python -m ipykernel install --user --name=genbio --display-name="GenBio-AI (scFoundation/AIDOCell)
```

In [1]:
import os
import logging
from pathlib import Path

import numpy as np

from napistu.genomics.scverse_loading import DatasetsConfig

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

DATA_DIR = "data"
OUTPUT_DIR = Path("~/Desktop/DATA/sc_foundation_models/model_outputs").expanduser()
# Raw config dictionary

DATASET_NAME = "efthymiou2025"

DATASETS_CONFIG = {
    DATASET_NAME: {
        "uri": "https://cellxgene.cziscience.com/collections/6b701826-37bb-4356-9792-ff41fc4c3161",
        "path": Path("~/Desktop/DATA/genomics/efthymiou.h5ad").expanduser()
    }
}

MODEL_PATH = os.path.join(DATA_DIR, "genbioAI")
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Validated config
datasets_config = DatasetsConfig(DATASETS_CONFIG)

# Set environmental variables so models are downloaded to the the model path rather than ~/.cache/huggingface
os.environ['HF_HOME'] = MODEL_PATH                        # Primary control
os.environ['HUGGINGFACE_HUB_CACHE'] = MODEL_PATH          # Hub downloads
os.environ['TRANSFORMERS_CACHE'] = MODEL_PATH             # Transformers-specific

from napistu_torch.foundation_models.etl import (
    process_aidocell,
    process_scfoundation,
)
from napistu_torch.foundation_models.foundation_models import FoundationModel
from napistu_torch.foundation_models.constants import (
    AIDOCELL_CLASSES,
    FOUNDATION_MODEL_NAMES,
)


## AIDOCell

Format each model as a `napistu_torch.load.foundation_model.FoundationModel` instance plust a metadata json and save to disk.

In [ ]:
CELLS_PER_MODEL = {
    AIDOCELL_CLASSES.THREE_M : 100,
    AIDOCELL_CLASSES.TEN_M : 100,
    AIDOCELL_CLASSES.ONE_HUNDRED_M : 10
}

# Process all AIDOCell variants using class names from AIDOCELL_CLASSES
for class_name in [AIDOCELL_CLASSES.THREE_M, AIDOCELL_CLASSES.TEN_M, AIDOCELL_CLASSES.ONE_HUNDRED_M]:
    try:
        process_aidocell(class_name, OUTPUT_DIR, datasets_config=datasets_config, cells_per_cluster=CELLS_PER_MODEL[class_name])
    except Exception as e:
        print(f"Error processing {class_name}: {e}")
        continue


In [ ]:
for class_name in [AIDOCELL_CLASSES.THREE_M, AIDOCELL_CLASSES.TEN_M, AIDOCELL_CLASSES.ONE_HUNDRED_M]:
    try:    
        prefix = f"{FOUNDATION_MODEL_NAMES.AIDOCELL}_{class_name}"

        # Load model using FoundationModel.load()
        foundation_model = FoundationModel.load(OUTPUT_DIR, prefix)    
    except Exception as e:
        logger.error(f"Error processing {class_name}: {e}")
        continue


## scFoundation


In [3]:
# End-to-end smoke: write artefacts under OUTPUT_DIR then reload.
CELLS_PER_CLUSTER = 50
MIN_CELLS_PER_GENE_EMB = 2

process_scfoundation(
    output_dir=str(OUTPUT_DIR),
    cache_dir=MODEL_PATH,
    datasets_config=datasets_config,
    cells_per_cluster=CELLS_PER_CLUSTER,
    min_cells_per_gene_embedding=MIN_CELLS_PER_GENE_EMB,
)

foundation_model = FoundationModel.load(
    OUTPUT_DIR / FOUNDATION_MODEL_NAMES.SCFOUNDATION
)
out_path = OUTPUT_DIR / FOUNDATION_MODEL_NAMES.SCFOUNDATION
print(f"saved + loaded ({type(foundation_model).__name__}) from {out_path}")


INFO:napistu_torch.foundation_models.etl:Extracting: scFoundation
INFO:napistu_torch.foundation_models.etl:
1. Downloading checkpoint from HuggingFace...
INFO:napistu_torch.foundation_models.etl:Loading scFoundation checkpoint (gene encoder: gene)
INFO:napistu_torch.foundation_models.etl:2. Extracting weights...


• standardized 19208/19264 terms


INFO:napistu_torch.foundation_models.etl:Extracting model weights...
INFO:napistu_torch.foundation_models.etl:  ✓ Extracted gene embeddings: (19264, 768)
INFO:napistu_torch.foundation_models.etl:  ✓ Extracted 12 attention layers
INFO:napistu_torch.foundation_models.etl:Extracting metadata...
INFO:napistu_torch.foundation_models.etl:   19264 genes, 12 layers
INFO:napistu_torch.foundation_models.etl:   Embeddings: 768
INFO:napistu_torch.foundation_models.etl:   Attention weights: 12 layers × 4 matrices (Q,K,V,O)
INFO:napistu_torch.foundation_models.etl:3. Extracting dataset expression embeddings...
INFO:napistu_torch.foundation_models.etl:Excluded 1 cluster(s) with < 10 cells
INFO:napistu_torch.foundation_models.etl:Cluster 0: 12361 cells -> 50 cells in forward pass (cells_per_cluster=50)
INFO:napistu_torch.foundation_models.etl:  ✓ Completed cluster 0 vocabulary 9176/19015 genes retained (50 cells forwarded, finite frac≈1.000)
INFO:napistu_torch.foundation_models.etl:Cluster 1: 11041 ce

saved + loaded (FoundationModel) from /Users/sean/Desktop/DATA/sc_foundation_models/model_outputs/scFoundation
